#### Objetivo: Este código de teste para aplicar varmax adicionando manualmente o número de lagas para teste, grau de polinômios (exógenas e endógena) e também tenta adicionar às influências das partes reais e imaginárias

In [1]:
# ---------------------
# Incluindo Bibliotecas
# ---------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VARMAX
from statsmodels.tsa.stattools import coint,kpss
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from itertools import product,combinations

In [2]:
# ---------------------------------------
# Função de Criação Automática de Lag (m)
# ---------------------------------------
def select_auto_lag(df_exog,exog_cols,max_x_lag=2,inplace=True) -> pd.DataFrame:
  """
  Aplica os lags desejado às exógenas da base de dados original e retorna o dataframe com as novas colunas
  """
  new_exog_cols = list(exog_cols)

  for lag in range(1, max_x_lag + 1):
    for col in exog_cols:
      lag_cols_name = f'{col}_lag{lag}'
      df_exog[lag_cols_name] = df_exog[col].shift(lag)
      new_exog_cols.append(lag_cols_name)

  lag_cols = [f'{col}_lag{lag}' for lag in range(1, max_x_lag+1) for col in exog_cols]
  rows_to_drop = df_exog.index[df_exog[lag_cols].isna().any(axis=1)]

  df_exog = df_exog.drop(index=rows_to_drop)
  return df_exog,new_exog_cols

In [3]:
# -------------------------------------------------
# Função de Seleção de grau de Polinômio de Lag (m)
# -------------------------------------------------
def select_auto_polinomial_degree(df_exog,exog_cols,degree_type,max_degree=1) -> pd.DataFrame:
  """
  Aplica os graus desejado às exógenas da base de dados original, gerando o polinômio completo para cada coluna
  das exógenas (originais e sem lags) e retorna o dataframe com as novas colunas
  """
  new_exog_cols = list(exog_cols)
  new_columns = {}

  if degree_type in ['real','r','REAL','Real']:
    degree_type = 'real'
  elif degree_type in ['img','i','IMG','Img']:
    degree_type = 'img'
  else:
    raise ValueError("degree_type must be 'real' or 'img'")

  # Aplicando graus para série de dados reais
  if max_degree >= 2:
    for degree in range(2, max_degree + 1):
      columns = [col for col in exog_cols if degree_type in col]
      for col in columns:
        degree_col_name = f'{col}_deg{degree}'
        new_columns[degree_col_name] = df_exog[col] ** degree
        new_exog_cols.append(degree_col_name)

  if new_columns:
    df_exog = pd.concat([df_exog,pd.DataFrame(new_columns,index=df_exog.index)],axis=1)

  return df_exog,new_exog_cols

In [5]:
# -----------------------------------------
# Função de Forecast e Cálculo das Métricas
# -----------------------------------------
def get_forecast_metrics(p, res, val, endog_cols, current_exog_cols) -> dict:
    """
    Calcula as métricas de avaliação da previsão realizada pelo modelo
    """
    fc = res.forecast(steps=len(val[list(endog_cols)]), exog=val[current_exog_cols])
    metrics = {}

    for col in endog_cols:
        y_true = val[col]
        y_pred = fc[col]

        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        adj_r2 = 1 - (1 - r2) * (len(y_true) - 1) / (len(y_true) - p - 1)
        std_err = np.std(y_true - y_pred)

        metrics[col] = {
          'MSE':    mse,
          'RMSE':   rmse,
          'MAE':    mae,
          'R2':     r2,
          'AdjR2':  adj_r2,
          'STD':    std_err,
        }

    # Métricas globais do modelo
    metrics['rmse_mean'] = np.mean([metrics[col]['RMSE'] for col in endog_cols])
    metrics['aic'] = getattr(res, 'aic', np.nan)
    metrics['bic'] = getattr(res, 'bic', np.nan)

    return metrics

In [6]:
# -----------------------------
# Função de Impressão dos Dados
# -----------------------------
def view_result(data,endog_cols=('Yreal', 'Yimg')):
    """
    Imprime na tela as métricas calculadas de acordo com a previsão do modelo
    """
    metric_labels = [
        ('RMSE',  'RMSE'),
        ('MAE',   'MAE'),
        ('R2',    'R²'),
        ('AdjR2', 'R² Ajust'),
        ('STD',   'STD')]

    W_METRIC = 10
    W_VAL    = 10

    def sep(l, m, r, n_cols):
        block = '─' * (W_METRIC + W_VAL + 3)
        return l + m.join([block] * n_cols) + r

    def row_str(pairs):
        cells = [f' {lbl:<{W_METRIC}} {val:>{W_VAL}} ' for lbl, val in pairs]
        return '│' + '│'.join(cells) + '│'

    print(f'Lag={data["p"]}')
    print(f'  AIC={data["aic"]:.2f} | BIC={data["bic"]:.2f} | RMSE Mean={data["rmse_mean"]:.4f}')
    print(f'  X Lag={data["x_lag"]} | Exog Degree={data["exog_degree"]}')

    print(sep('  ┌', '┬', '┐', len(endog_cols)))
    print('  ' + row_str([(col, '') for col in endog_cols]))
    print(sep('  ├', '┼', '┤', len(endog_cols)))

    for key, lbl in metric_labels:
        pairs = []
        for col in endog_cols:
            val = data[col][key]
            v_str = f'{val:+.4f}' if key in ('R2', 'AdjR2') else f'{val:.4f}'
            pairs.append((lbl, v_str))
        print('  ' + row_str(pairs))

    print(sep('  └', '┴', '┘', len(endog_cols)))

In [7]:
# -------------------------------------
# Função de Seleção de Teste de Lag (m)
# -------------------------------------
def calc_train_sect(data:pd.DataFrame,train_frac:float,val_frac:float):
  """
  Calcula o tamanho dos conjuntos de treino, validação e teste
  """
  n = len(data)
  train_end = int(n*train_frac)
  val_end = int(n*(train_frac + val_frac))
  return train_end,val_end

def gridsearch_custom_varmax(
    data:pd.DataFrame,
    endog_cols:list,
    exog_cols:list,
    max_p:int=3,
    max_x_lag:int=2,
    train_frac:float=0.6,
    val_frac:float=0.2,
    endog_real_degree:int=0,
    endog_img_degree:int=0,
    exog_real_degree:int=0,
    exog_img_degree:int=0,
    trends:list=['c'],
    verbose:bool=True) -> pd.DataFrame:
  """
  Busca a melhor combinação de hiperparâmetros para o modelo VARMAX com base nas métricas de avaliação (RMSE,AIC,BIC)
  """
  train_end,val_end = calc_train_sect(data,train_frac,val_frac)
  results = []

  for p in range(1, max_p + 1):
    for x_lag in range(0, max_x_lag + 1):
      for tr in trends:
        df_temp = data.copy()
        current_exog_cols = list(exog_cols)

        if x_lag > 0:
          df_temp,current_exog_cols = select_auto_lag(df_temp,exog_cols,max_x_lag=x_lag)

        degree_combination = list(product(range(1,exog_real_degree+1),range(1,exog_img_degree+1)))

        if not degree_combination and (exog_real_degree == 0 or exog_img_degree == 0):
            degree_combination = [(1, 1)]

        for real,img in degree_combination:
          _df_temp = df_temp.copy()
          train_exog_cols = list(current_exog_cols)

          if real >= 2:
            _df_temp, train_exog_cols = select_auto_polinomial_degree(_df_temp,train_exog_cols,'r',max_degree=real)

          if img >= 2:
            _df_temp, train_exog_cols = select_auto_polinomial_degree(_df_temp,train_exog_cols,'i',max_degree=img)

          train = _df_temp.iloc[:train_end].copy()
          val = _df_temp.iloc[train_end:val_end].copy()
          test = _df_temp.iloc[val_end:].copy()

          try:
            model = VARMAX(
                    train[list(endog_cols)],
                    exog=train[train_exog_cols],
                    order=(p, 0),
                    trend=tr)
            res = model.fit(disp=False,maxiter=500,method='powell')
            dict_res = get_forecast_metrics(p,res,val,endog_cols,train_exog_cols)
            dict_res.update({'p':p,'x_lag':x_lag,'exog_degree':f"(r:{real},i:{img})",'trend':tr})

            if verbose: view_result(dict_res)

          except Exception as e:
            print(f"Erro ao treinar o modelo: {e}")
            continue

  return pd.DataFrame(results)

#### Leitura dos dataframes iniciais

In [8]:
data = "dadosIniciais.xlsx"
data = pd.read_excel(data)

In [9]:
df_input = pd.DataFrame()
df_input["Xreal"] = data.iloc[:,0]
df_input["Ximg"] = data.iloc[:,1]

In [10]:
df_output = pd.DataFrame()
df_output["Yreal"] = data.iloc[:,2]
df_output["Yimg"] = data.iloc[:,3]

In [11]:
df_concat = pd.concat([df_output,df_input],axis=1)

#### Teste de geração de lags

In [12]:
current_exog_cols = ["Xreal","Ximg"]
cp_df_input = df_input.copy()
cp_df_input, current_exog_cols = select_auto_lag(cp_df_input,current_exog_cols,max_x_lag=5,inplace=False)

print(f"current_exog_cols: {current_exog_cols}")

current_exog_cols: ['Xreal', 'Ximg', 'Xreal_lag1', 'Ximg_lag1', 'Xreal_lag2', 'Ximg_lag2', 'Xreal_lag3', 'Ximg_lag3', 'Xreal_lag4', 'Ximg_lag4', 'Xreal_lag5', 'Ximg_lag5']


#### Teste de Geração de Graus de Polinômio

In [13]:
exog_cols = cp_df_input.columns
current_exog_cols = list(exog_cols)
cp_df_input, current_exog_cols = select_auto_polinomial_degree(cp_df_input,exog_cols,'r',max_degree=3)

print(f"{[col for col in list(cp_df_input.columns) if 'X' in col]}")
print(f"current_exog_cols: {current_exog_cols}")

['Xreal', 'Ximg', 'Xreal_lag1', 'Ximg_lag1', 'Xreal_lag2', 'Ximg_lag2', 'Xreal_lag3', 'Ximg_lag3', 'Xreal_lag4', 'Ximg_lag4', 'Xreal_lag5', 'Ximg_lag5', 'Xreal_deg2', 'Xreal_lag1_deg2', 'Xreal_lag2_deg2', 'Xreal_lag3_deg2', 'Xreal_lag4_deg2', 'Xreal_lag5_deg2', 'Xreal_deg3', 'Xreal_lag1_deg3', 'Xreal_lag2_deg3', 'Xreal_lag3_deg3', 'Xreal_lag4_deg3', 'Xreal_lag5_deg3']
current_exog_cols: ['Xreal', 'Ximg', 'Xreal_lag1', 'Ximg_lag1', 'Xreal_lag2', 'Ximg_lag2', 'Xreal_lag3', 'Ximg_lag3', 'Xreal_lag4', 'Ximg_lag4', 'Xreal_lag5', 'Ximg_lag5', 'Xreal_deg2', 'Xreal_lag1_deg2', 'Xreal_lag2_deg2', 'Xreal_lag3_deg2', 'Xreal_lag4_deg2', 'Xreal_lag5_deg2', 'Xreal_deg3', 'Xreal_lag1_deg3', 'Xreal_lag2_deg3', 'Xreal_lag3_deg3', 'Xreal_lag4_deg3', 'Xreal_lag5_deg3']


In [14]:
real_exog_cols = [col for col in current_exog_cols if 'real' in col]
img_exog_cols = [col for col in current_exog_cols if 'img' in col]

print(f"real_exog_cols: {real_exog_cols}")
print(f"img_exog_cols: {img_exog_cols}")

real_exog_cols: ['Xreal', 'Xreal_lag1', 'Xreal_lag2', 'Xreal_lag3', 'Xreal_lag4', 'Xreal_lag5', 'Xreal_deg2', 'Xreal_lag1_deg2', 'Xreal_lag2_deg2', 'Xreal_lag3_deg2', 'Xreal_lag4_deg2', 'Xreal_lag5_deg2', 'Xreal_deg3', 'Xreal_lag1_deg3', 'Xreal_lag2_deg3', 'Xreal_lag3_deg3', 'Xreal_lag4_deg3', 'Xreal_lag5_deg3']
img_exog_cols: ['Ximg', 'Ximg_lag1', 'Ximg_lag2', 'Ximg_lag3', 'Ximg_lag4', 'Ximg_lag5']


#### Teste da Seleção de Lags do Varmax

In [ ]:
endog_cols = ["Yreal","Yimg"]
exog_cols = ["Xreal","Ximg"]

res = gridsearch_custom_varmax(
    data=df_concat,
    endog_cols=endog_cols,
    exog_cols=exog_cols,
    max_p=3,
    exog_real_degree=2,
    exog_img_degree=2)

view_result(res.sort_values(by="rmse"))

Lag=1
  AIC=31296.76 | BIC=31404.35 | RMSE Mean=12.3026
  X Lag=0 | Exog Degree=(r:1,i:1)
  ┌───────────────────────┬───────────────────────┐
  │ Yreal                 │ Yimg                  │
  ├───────────────────────┼───────────────────────┤
  │ RMSE          12.6395 │ RMSE          11.9658 │
  │ MAE           10.3440 │ MAE            9.7990 │
  │ R²            +0.0557 │ R²            +0.1529 │
  │ R² Ajust      +0.0556 │ R² Ajust      +0.1528 │
  │ STD           12.6395 │ STD           11.9658 │
  └───────────────────────┴───────────────────────┘
Lag=1
  AIC=31303.31 | BIC=31427.45 | RMSE Mean=12.2839
  X Lag=0 | Exog Degree=(r:1,i:2)
  ┌───────────────────────┬───────────────────────┐
  │ Yreal                 │ Yimg                  │
  ├───────────────────────┼───────────────────────┤
  │ RMSE          12.6291 │ RMSE          11.9386 │
  │ MAE           10.3100 │ MAE            9.7524 │
  │ R²            +0.0572 │ R²            +0.1568 │
  │ R² Ajust      +0.0571 │ R² Ajust    